In [ ]:
# 数据来源于墨尔本的房屋价格
import pandas as pd
from sklearn.tree import DecisionTreeRegressor

file_path = "./melb_data.csv"
data = pd.read_csv(file_path)
data = data.dropna(axis=0)
y = data.Price
data_features = ['Rooms', 'Bathroom', 'Landsize', 'BuildingArea', 'YearBuilt', 'Lattitude', 'Longtitude']
x = data[data_features]
model = DecisionTreeRegressor(random_state=1)
model.fit(x, y)

# 尝试简单的模型预测
print("Making predictions for the following 5 houses:")
print(x.head())
print("The predictions are")
print(model.predict(x.head()))

Making predictions for the following 5 houses:
   Rooms  Bathroom  Landsize  BuildingArea  YearBuilt  Lattitude  Longtitude
1      2       1.0     156.0          79.0     1900.0   -37.8079    144.9934
2      3       2.0     134.0         150.0     1900.0   -37.8093    144.9944
4      4       1.0     120.0         142.0     2014.0   -37.8072    144.9941
6      3       2.0     245.0         210.0     1910.0   -37.8024    144.9993
7      2       1.0     256.0         107.0     1890.0   -37.8060    144.9954
The predictions are
[1035000. 1465000. 1600000. 1876000. 1636000.]


In [11]:
predicted_home_prices = model.predict(x)

# 计算平均绝对误差
from sklearn.metrics import mean_absolute_error
print(mean_absolute_error(y, predicted_home_prices))

434.71594577146544


In [18]:
from sklearn.model_selection import train_test_split

# 为了验证模型的泛化能力，我们需要将数据划分为训练集和验证集
train_x, test_x, train_y, test_y = train_test_split(x, y, random_state=0)
model = DecisionTreeRegressor(random_state=0)
model.fit(train_x, train_y)
predicted_home_prices = model.predict(test_x)

# 计算平均绝对误差
from sklearn.metrics import mean_absolute_error
print(mean_absolute_error(test_y, predicted_home_prices))

265345.3511943189


In [22]:
# 使用不同数量叶子节点的决策树回归模型训练
# 超出过拟合和欠拟合的平衡点
from tqdm import trange

def find_best(leaf_nodes, train_x, train_y, test_x, test_y):
    model = DecisionTreeRegressor(max_leaf_nodes=leaf_nodes, random_state=0)
    model.fit(train_x, train_y)
    predicted_home_prices = model.predict(test_x)
    return mean_absolute_error(test_y, predicted_home_prices)

best_mae = 10000000
best_leaf_nodes = 0
for nodes in trange(5, 1000):
    mae = find_best(nodes, train_x, train_y, test_x, test_y)
    if mae < best_mae :
        best_mae = mae
        best_leaf_nodes = nodes

print("Best leaf nodes:", best_leaf_nodes)
print("Best Mean Absolute Error:", best_mae)    

100%|██████████| 995/995 [00:16<00:00, 59.72it/s]

Best leaf nodes: 367
Best Mean Absolute Error: 241027.95803757792


In [ ]:
# 看得出来，使用决策树模型的学习结果并不好，最优情况下的平均绝对误差也有240000左右
# 因此我们考虑使用随机森林进行机器学习，进而获得更好的预测结果
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

file_path = "./melb_data.csv"
data = pd.read_csv(file_path)
data = data.dropna(axis=0)
y = data.Price
data_features = ['Rooms', 'Bathroom', 'Landsize', 'BuildingArea', 'YearBuilt', 'Lattitude', 'Longtitude']
x = data[data_features]

train_x, test_x, train_y, test_y = train_test_split(x, y, random_state=0)
new_model = RandomForestRegressor(random_state=0)
new_model.fit(train_x, train_y)
new_predicted_home_prices = new_model.predict(test_x)
print(mean_absolute_error(test_y, new_predicted_home_prices))

193528.28707553257
